# Daily Challenge — Building BaristaBot with LangGraph and Gemini

## Complete and heavily documented notebook

BaristaBot is a stateful café-ordering agent built with Gemini, LangChain,
and LangGraph.

The completed application can:

- answer questions about a live menu;
- add drinks and modifiers to a structured order;
- show, clear, confirm, and place the order;
- loop between the customer, the model, and tools;
- stop when the customer quits or the order is placed.

Every major block is explained in Markdown and commented directly in code.

## Learning objectives

You will learn how to:

1. define a LangGraph state with `TypedDict`;
2. append messages with the `add_messages` reducer;
3. create nodes that return partial state updates;
4. connect nodes with fixed and conditional edges;
5. bind Python tools to Gemini;
6. execute stateless tools through `ToolNode`;
7. handle state-changing tools in a custom node;
8. build a human-in-the-loop conversation;
9. visualize and test a graph;
10. keep model decisions separate from trusted state changes.

## Final graph architecture

```text
                     START
                       ↓
                    chatbot
                ┌──────┼──────┐
                ↓      ↓      ↓
              tools ordering human
                │      │       │
                └──────┴──→ chatbot
                               │
                    quit/order placed
                               ↓
                              END
```

- `chatbot` decides whether to speak or call a tool.
- `tools` executes the stateless live-menu tool.
- `ordering` safely updates application state.
- `human` collects the next user message.

# 1. Install the required packages

In [ ]:
# Pin the versions requested by the assignment.
#
# `%pip` installs packages into the active notebook runtime.
# `-qU` means quiet output and upgrade when needed.

%pip install -qU \
    "langgraph==1.0.5" \
    "langchain-google-genai==4.1.2" \
    "google-genai==1.56.0" \
    "typing-extensions>=4.12"

If an older LangChain or LangGraph package was already imported, restart the
runtime once after installation and run all cells again.

In [ ]:
# Standard-library imports.
import getpass
import importlib.metadata as metadata
import os
from pprint import pprint
from random import randint
from typing import Annotated, Literal

# Typing helper used for the graph state schema.
from typing_extensions import TypedDict

# Jupyter display helpers for graph images.
from IPython.display import Image, display

# LangChain message and tool abstractions.
from langchain_core.messages import (
    AIMessage,
    AnyMessage,
    HumanMessage,
    ToolMessage,
)
from langchain_core.tools import tool

# Gemini chat integration.
from langchain_google_genai import ChatGoogleGenerativeAI

# LangGraph graph primitives.
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("Installed versions:")
for package_name in [
    "langgraph",
    "langchain-google-genai",
    "google-genai",
]:
    try:
        print(f"- {package_name}: {metadata.version(package_name)}")
    except metadata.PackageNotFoundError:
        print(f"- {package_name}: not found")

# 2. Configure the Gemini API key

The helper supports an existing environment variable, a Kaggle secret, a
Colab secret, and a hidden manual prompt. The key is never printed.

In [ ]:
def configure_google_api_key() -> str:
    """Configure GOOGLE_API_KEY without displaying its value."""

    # Reuse an existing environment variable.
    if os.getenv("GOOGLE_API_KEY"):
        return "existing environment variable"

    # Kaggle: Add-ons → Secrets → GOOGLE_API_KEY.
    try:
        from kaggle_secrets import UserSecretsClient

        kaggle_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
        if kaggle_key:
            os.environ["GOOGLE_API_KEY"] = kaggle_key
            return "Kaggle secret"
    except Exception:
        # Expected outside Kaggle.
        pass

    # Google Colab: Secrets panel → GOOGLE_API_KEY.
    try:
        from google.colab import userdata

        colab_key = userdata.get("GOOGLE_API_KEY")
        if colab_key:
            os.environ["GOOGLE_API_KEY"] = colab_key
            return "Google Colab secret"
    except Exception:
        # Expected outside Colab.
        pass

    # Local/Jupyter fallback.
    manual_key = getpass.getpass(
        "Enter your GOOGLE_API_KEY: "
    ).strip()

    if not manual_key:
        raise RuntimeError("No Gemini API key was supplied.")

    os.environ["GOOGLE_API_KEY"] = manual_key
    return "manual hidden input"


API_KEY_SOURCE = configure_google_api_key()
print("Gemini API key configured from:", API_KEY_SOURCE)

# 3. Define state and system instructions

LangGraph passes one shared state object between nodes.

`messages` uses the `add_messages` reducer, so new messages are appended.
`order` and `finished` use normal replacement behavior.

In [ ]:
class OrderState(TypedDict):
    """State shared by every BaristaBot node."""

    # Conversation history. LangGraph appends new messages through
    # the add_messages reducer instead of replacing the full list.
    messages: Annotated[list[AnyMessage], add_messages]

    # Current order represented as readable strings.
    order: list[str]

    # True after the customer quits or the order is placed.
    finished: bool


BARISTABOT_SYSINT = (
    "system",
    """
    You are BaristaBot, an interactive cafe ordering assistant.

    Scope:
    - Discuss only the cafe menu, drinks, modifiers, and their history.
    - Politely redirect off-topic requests back to ordering.

    Tool rules:
    - Call get_menu when current menu or stock information is needed.
    - Call add_to_order for every accepted drink.
    - Call get_order when you need the current order.
    - Call clear_order only when the customer wants to restart.
    - Always call confirm_order before place_order.
    - After confirmation, apply requested changes before placing.
    - Never call menu tools and order-changing tools in the same response.
    - Never invent drinks or modifiers.
    - Soy milk is unavailable today.
    - Do not claim state changed without calling a tool.

    Completion:
    - After place_order returns, thank the customer, mention the estimated
      preparation time, and say goodbye.
    """.strip(),
)


WELCOME_MSG = (
    "Welcome to the BaristaBot cafe. "
    "Type `q` to quit. How may I serve you today?"
)


def initial_order_state(
    messages: list | None = None,
) -> OrderState:
    """Create a complete state with safe defaults."""
    return {
        "messages": messages or [],
        "order": [],
        "finished": False,
    }

## Why the model does not edit state directly

Gemini may request actions through tool calls, but trusted Python code
validates and performs the actual order updates. This prevents arbitrary
model-generated state mutations.

In [ ]:
# Use an explicit stable model name instead of a moving "latest" alias.
# Set GEMINI_MODEL in the environment to test another compatible model.
GEMINI_MODEL = os.getenv(
    "GEMINI_MODEL",
    "gemini-2.5-flash",
)

# Temperature 0 improves consistency for routing and tool calls.
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0,
)

print("Gemini model configured:", GEMINI_MODEL)

# 4. Build a one-turn chatbot graph

In [ ]:
def chatbot(state: OrderState) -> dict:
    """Call Gemini once and append its response."""

    # Prepend the system instruction to the accumulated conversation.
    message_history = [
        BARISTABOT_SYSINT,
        *state["messages"],
    ]

    # Gemini returns an AIMessage.
    model_message = llm.invoke(message_history)

    # Return only the new message.
    return {"messages": [model_message]}

In [ ]:
# Create the graph and register the chatbot node.
one_turn_builder = StateGraph(OrderState)
one_turn_builder.add_node("chatbot", chatbot)

# Fixed transitions: START → chatbot → END.
one_turn_builder.add_edge(START, "chatbot")
one_turn_builder.add_edge("chatbot", END)

# Compile before invocation.
chat_graph = one_turn_builder.compile()

print("One-turn graph compiled.")

In [ ]:
def display_graph(graph, title: str) -> None:
    """Display a PNG, or Mermaid text when PNG rendering is unavailable."""

    print(title)

    try:
        png_bytes = graph.get_graph().draw_mermaid_png()
        display(Image(png_bytes))
    except Exception as error:
        print("PNG rendering unavailable; showing Mermaid source.")
        print(graph.get_graph().draw_mermaid())
        print("Rendering detail:", type(error).__name__)


display_graph(chat_graph, "One-turn chatbot graph")

In [ ]:
# Run one turn.
user_msg = "What drinks can I order?"

state = chat_graph.invoke(
    initial_order_state([
        HumanMessage(content=user_msg)
    ])
)

for message in state["messages"]:
    print(f"{type(message).__name__}: {message.content}")

## Manually invoke a second turn

The first final state is reused. A new `HumanMessage` is appended before
invoking the graph again.

In [ ]:
second_user_msg = "Do you have any tea with milk?"

second_turn_messages = [
    *state["messages"],
    HumanMessage(content=second_user_msg),
]

second_turn_state = {
    **state,
    "messages": second_turn_messages,
}

state = chat_graph.invoke(second_turn_state)

for message in state["messages"]:
    print(f"{type(message).__name__}: {message.content}")

# 5. Add a human node and a conversation loop

In [ ]:
QUIT_WORDS = {
    "q",
    "quit",
    "exit",
    "goodbye",
}


def human_node(state: OrderState) -> dict:
    """Display the last assistant message and collect user input."""

    last_message = state["messages"][-1]
    print("Model:", last_message.content)

    # A real app could replace input() with a web form or microphone.
    user_input = input("User: ").strip()

    wants_to_exit = user_input.lower() in QUIT_WORDS

    return {
        "messages": [
            HumanMessage(content=user_input)
        ],
        "finished": (
            state.get("finished", False)
            or wants_to_exit
        ),
    }


def chatbot_with_welcome_msg(
    state: OrderState,
) -> dict:
    """Welcome on the first cycle, then use Gemini."""

    if state.get("messages"):
        model_message = llm.invoke([
            BARISTABOT_SYSINT,
            *state["messages"],
        ])
    else:
        # Avoid an API call for the initial welcome.
        model_message = AIMessage(
            content=WELCOME_MSG
        )

    return {"messages": [model_message]}


def maybe_exit_human_node(
    state: OrderState,
) -> Literal["chatbot", "__end__"]:
    """Continue chatting unless the user has finished."""

    if state.get("finished", False):
        return END

    return "chatbot"

In [ ]:
loop_builder = StateGraph(OrderState)

loop_builder.add_node(
    "chatbot",
    chatbot_with_welcome_msg,
)
loop_builder.add_node(
    "human",
    human_node,
)

loop_builder.add_edge(START, "chatbot")
loop_builder.add_edge("chatbot", "human")

# Conditional edge: human → chatbot or END.
loop_builder.add_conditional_edges(
    "human",
    maybe_exit_human_node,
)

chat_with_human_graph = loop_builder.compile()

display_graph(
    chat_with_human_graph,
    "Human-in-the-loop graph",
)

In [ ]:
# Disabled by default so "Run all" does not block at input().
RUN_BASIC_INTERACTIVE_LOOP = False

if RUN_BASIC_INTERACTIVE_LOOP:
    basic_loop_state = chat_with_human_graph.invoke(
        initial_order_state(),
        {"recursion_limit": 50},
    )
    pprint(basic_loop_state)
else:
    print(
        "Set RUN_BASIC_INTERACTIVE_LOOP=True "
        "to start the basic loop."
    )

# 6. Add a live menu as a stateless tool

In [ ]:
# Canonical names are kept in application code for later validation.
VALID_DRINKS = {
    "espresso",
    "americano",
    "cold brew",
    "latte",
    "cappuccino",
    "cortado",
    "macchiato",
    "mocha",
    "flat white",
    "english breakfast tea",
    "green tea",
    "earl grey",
    "chai latte",
    "matcha latte",
    "london fog",
    "steamer",
    "hot chocolate",
}


MENU_TEXT = """
MENU

Coffee Drinks:
- Espresso
- Americano
- Cold Brew

Coffee Drinks with Milk:
- Latte
- Cappuccino
- Cortado
- Macchiato
- Mocha
- Flat White

Tea Drinks:
- English Breakfast Tea
- Green Tea
- Earl Grey

Tea Drinks with Milk:
- Chai Latte
- Matcha Latte
- London Fog

Other Drinks:
- Steamer
- Hot Chocolate

Modifiers:
- Milk: Whole, 2%, Oat, Almond, 2% Lactose Free
- Espresso shots: Single, Double, Triple, Quadruple
- Caffeine: Decaf or Regular
- Temperature: Hot or Iced
- Sweeteners: vanilla, hazelnut, caramel sauce, chocolate sauce,
  sugar-free vanilla
- Reasonable requests: extra hot, one pump, half caff, extra foam

Notes:
- "Dirty" adds an espresso shot.
- "Regular milk" means whole milk.
- "Sweetened" means regular sugar.
- Soy milk is out of stock today.
""".strip()


@tool
def get_menu() -> str:
    """Return the latest menu and stock information."""

    # A production version could read a database or inventory API.
    return MENU_TEXT

In [ ]:
# ToolNode automatically executes stateless tools.
menu_tools = [get_menu]
menu_tool_node = ToolNode(menu_tools)

# Binding exposes the get_menu schema to Gemini.
llm_with_menu_tools = llm.bind_tools(menu_tools)


def chatbot_with_menu_tools(
    state: OrderState,
) -> dict:
    """Chatbot variant that can request the menu tool."""

    if state.get("messages"):
        model_message = llm_with_menu_tools.invoke([
            BARISTABOT_SYSINT,
            *state["messages"],
        ])
    else:
        model_message = AIMessage(
            content=WELCOME_MSG
        )

    return {
        "messages": [model_message],
        "order": state.get("order", []),
        "finished": state.get("finished", False),
    }


def maybe_route_to_menu_tool(
    state: OrderState,
) -> Literal["tools", "human"]:
    """Route a Gemini tool call to ToolNode."""

    messages = state.get("messages", [])
    if not messages:
        raise ValueError("No messages were found in state.")

    last_message = messages[-1]

    if getattr(last_message, "tool_calls", []):
        return "tools"

    return "human"

In [ ]:
menu_builder = StateGraph(OrderState)

menu_builder.add_node(
    "chatbot",
    chatbot_with_menu_tools,
)
menu_builder.add_node(
    "human",
    human_node,
)
menu_builder.add_node(
    "tools",
    menu_tool_node,
)

menu_builder.add_edge(START, "chatbot")

# chatbot → tools or human
menu_builder.add_conditional_edges(
    "chatbot",
    maybe_route_to_menu_tool,
)

# Tool result returns to Gemini.
menu_builder.add_edge("tools", "chatbot")

# Human continues or exits.
menu_builder.add_conditional_edges(
    "human",
    maybe_exit_human_node,
)

graph_with_menu = menu_builder.compile()

display_graph(
    graph_with_menu,
    "BaristaBot with live-menu tool",
)

# 7. Define order-changing tool schemas

Gemini can see and request these tools, but they are not placed in the
automatic `ToolNode`. A trusted custom node performs the real state updates.

In [ ]:
@tool
def add_to_order(
    drink: str,
    modifiers: list[str],
) -> str:
    """Add one menu drink and its modifiers to the current order."""
    return "Handled by order_node."


@tool
def confirm_order() -> str:
    """Display the exact order and ask the customer to confirm it."""
    return "Handled by order_node."


@tool
def get_order() -> str:
    """Return the current order, one item per line."""
    return "Handled by order_node."


@tool
def clear_order() -> str:
    """Remove all items from the current order."""
    return "Handled by order_node."


@tool
def place_order() -> int:
    """Send a confirmed non-empty order to the kitchen and return ETA."""
    return 0


order_tools = [
    add_to_order,
    confirm_order,
    get_order,
    clear_order,
    place_order,
]

ORDER_TOOL_NAMES = {
    order_tool.name
    for order_tool in order_tools
}

print("Order tool schemas:", sorted(ORDER_TOOL_NAMES))

# 8. Implement the trusted order node

In [ ]:
def normalize_modifiers(raw_modifiers) -> list[str]:
    """Convert model-provided modifier data into a clean string list."""

    if raw_modifiers is None:
        return []

    if isinstance(raw_modifiers, str):
        raw_modifiers = [raw_modifiers]

    return [
        str(modifier).strip()
        for modifier in raw_modifiers
        if str(modifier).strip()
    ]


def order_node(state: OrderState) -> dict:
    """Validate tool calls and safely update order state."""

    messages = state.get("messages", [])
    if not messages:
        raise ValueError("order_node received no messages.")

    request_message = messages[-1]
    tool_calls = getattr(
        request_message,
        "tool_calls",
        [],
    )

    if not tool_calls:
        raise ValueError(
            "order_node expected at least one tool call."
        )

    # Copy the incoming list to avoid mutating the previous state object.
    order = list(state.get("order", []))

    outbound_messages: list[ToolMessage] = []
    order_placed = False

    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call["id"]

        if tool_name == "add_to_order":
            drink = str(
                tool_args.get("drink", "")
            ).strip()

            modifiers = normalize_modifiers(
                tool_args.get("modifiers", [])
            )

            # Validate the model-generated drink against application data.
            if drink.lower() not in VALID_DRINKS:
                response = (
                    f"{drink!r} is not a valid menu drink. "
                    "Ask the customer to choose an available item."
                )

            # Enforce today's stock rule in trusted code.
            elif any(
                "soy" in modifier.lower()
                for modifier in modifiers
            ):
                response = (
                    "Soy milk is unavailable today. "
                    "The item was not added."
                )

            else:
                modifier_text = (
                    ", ".join(modifiers)
                    if modifiers
                    else "no modifiers"
                )

                order.append(
                    f"{drink} ({modifier_text})"
                )

                response = (
                    "Updated order:\n"
                    + "\n".join(order)
                )

        elif tool_name == "confirm_order":
            # Print exact application state, not an LLM paraphrase.
            print("Your order:")

            if not order:
                print("  (no items)")

            for item in order:
                print(f"  {item}")

            response = input(
                "Is this correct? "
            ).strip()

        elif tool_name == "get_order":
            response = (
                "\n".join(order)
                if order
                else "(no order)"
            )

        elif tool_name == "clear_order":
            order.clear()
            response = "The order has been cleared."

        elif tool_name == "place_order":
            # Never place an empty order.
            if not order:
                response = (
                    "The order is empty and was not placed."
                )
            else:
                print("Sending order to kitchen!")
                print("\n".join(order))

                eta_minutes = randint(1, 5)
                order_placed = True

                response = (
                    "Order placed successfully. "
                    f"Estimated preparation time: "
                    f"{eta_minutes} minutes."
                )

        else:
            raise NotImplementedError(
                f"Unknown order tool: {tool_name}"
            )

        # Every tool call ID must receive a matching ToolMessage.
        outbound_messages.append(
            ToolMessage(
                content=str(response),
                name=tool_name,
                tool_call_id=tool_call_id,
            )
        )

    return {
        "messages": outbound_messages,
        "order": order,
        "finished": order_placed,
    }

# 9. Build routing for all tool groups

In [ ]:
# Stateless tools are executed automatically.
auto_tools = [get_menu]
final_menu_tool_node = ToolNode(auto_tools)

# Gemini must know every available schema.
all_tools = [
    *auto_tools,
    *order_tools,
]

llm_with_all_tools = llm.bind_tools(all_tools)


def maybe_route_to_tools(
    state: OrderState,
) -> Literal[
    "tools",
    "ordering",
    "human",
    "__end__",
]:
    """Choose the next node from the latest AI message."""

    messages = state.get("messages", [])
    if not messages:
        raise ValueError(
            "No messages were found while routing."
        )

    # After place_order, chatbot generates a final goodbye.
    # The next routing decision exits.
    if state.get("finished", False):
        return END

    last_message = messages[-1]
    tool_calls = getattr(
        last_message,
        "tool_calls",
        [],
    )

    if not tool_calls:
        return "human"

    requested_names = {
        tool_call["name"]
        for tool_call in tool_calls
    }

    automatic_names = set(
        final_menu_tool_node.tools_by_name
    )

    if requested_names <= automatic_names:
        return "tools"

    if requested_names <= ORDER_TOOL_NAMES:
        return "ordering"

    # Mixed groups can leave unmatched tool-call IDs.
    raise ValueError(
        "Unsupported or mixed tool groups: "
        f"{sorted(requested_names)}"
    )


def chatbot_with_all_tools(
    state: OrderState,
) -> dict:
    """Welcome first, then invoke Gemini with every tool schema."""

    if not state.get("messages"):
        model_message = AIMessage(
            content=WELCOME_MSG
        )
    else:
        model_message = llm_with_all_tools.invoke([
            BARISTABOT_SYSINT,
            *state["messages"],
        ])

    return {
        "messages": [model_message],
        "order": state.get("order", []),
        "finished": state.get("finished", False),
    }

# 10. Assemble the complete graph

In [ ]:
final_builder = StateGraph(OrderState)

final_builder.add_node(
    "chatbot",
    chatbot_with_all_tools,
)
final_builder.add_node(
    "human",
    human_node,
)
final_builder.add_node(
    "tools",
    final_menu_tool_node,
)
final_builder.add_node(
    "ordering",
    order_node,
)

# Entry point.
final_builder.add_edge(START, "chatbot")

# chatbot → tools, ordering, human, or END.
final_builder.add_conditional_edges(
    "chatbot",
    maybe_route_to_tools,
)

# human → chatbot or END.
final_builder.add_conditional_edges(
    "human",
    maybe_exit_human_node,
)

# Tool results always return to Gemini.
final_builder.add_edge("tools", "chatbot")
final_builder.add_edge("ordering", "chatbot")

graph_with_order_tools = final_builder.compile()

print("Complete BaristaBot graph compiled.")

In [ ]:
display_graph(
    graph_with_order_tools,
    "Complete BaristaBot graph",
)

# 11. Test the order node without calling Gemini

The following smoke tests construct synthetic tool calls. They verify
trusted state changes without consuming Gemini API quota.

In [ ]:
# Synthetic add_to_order call.
fake_add_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "add_to_order",
            "args": {
                "drink": "Latte",
                "modifiers": [
                    "Iced",
                    "Oat milk",
                ],
            },
            "id": "test-add-1",
            "type": "tool_call",
        }
    ],
)

add_test_state = order_node({
    "messages": [fake_add_message],
    "order": [],
    "finished": False,
})

print("State after add_to_order:")
pprint(add_test_state)

assert add_test_state["order"] == [
    "Latte (Iced, Oat milk)"
]
assert add_test_state["finished"] is False

In [ ]:
# Validate the unavailable-soy rule.
fake_soy_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "add_to_order",
            "args": {
                "drink": "Cappuccino",
                "modifiers": ["Soy milk"],
            },
            "id": "test-soy-1",
            "type": "tool_call",
        }
    ],
)

soy_test_state = order_node({
    "messages": [fake_soy_message],
    "order": [],
    "finished": False,
})

print("Soy validation result:")
pprint(soy_test_state)

assert soy_test_state["order"] == []

In [ ]:
# Verify that an empty order cannot be placed.
fake_empty_place_message = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "place_order",
            "args": {},
            "id": "test-place-empty-1",
            "type": "tool_call",
        }
    ],
)

empty_place_state = order_node({
    "messages": [fake_empty_place_message],
    "order": [],
    "finished": False,
})

print("Empty-order placement result:")
pprint(empty_place_state)

assert empty_place_state["finished"] is False

# 12. Run the complete interactive ordering system

In [ ]:
# A larger recursion limit allows several human/model/tool round trips.
config = {
    "recursion_limit": 100,
}

# Disabled by default so notebook "Run all" does not block at input().
RUN_FULL_INTERACTIVE_BARISTABOT = False

if RUN_FULL_INTERACTIVE_BARISTABOT:
    final_state = graph_with_order_tools.invoke(
        initial_order_state(),
        config,
    )

    print("\nFINAL STATE")
    pprint(final_state)
else:
    print(
        "Set RUN_FULL_INTERACTIVE_BARISTABOT=True "
        "to start ordering."
    )
    print(
        "Try asking for the menu, ordering a drink, "
        "changing it, confirming it, and placing it."
    )

## Suggested test scenarios

1. Ask: “What drinks are available?” and verify `get_menu`.
2. Order: “An iced oat latte.”
3. Request soy milk and verify that trusted code rejects it.
4. Clear the order and start again.
5. Confirm that the exact stored order is printed.
6. Try to place an empty order.
7. Enter `q`, `quit`, `exit`, or `goodbye`.

# 13. Design observations

## State

`messages` accumulates through a reducer. `order` and `finished` are replaced
through explicit node updates.

## Conditional routing

Routing functions inspect state and return the next node name. They do not
perform business logic.

## Tool separation

`get_menu` is stateless and safe for `ToolNode`. Order-changing actions use a
custom trusted node.

## Human-in-the-loop

The exact stored order is shown before confirmation, preventing the customer
from confirming a hallucinated paraphrase.

## Limitations

- `input()` is a notebook demo, not a production interface.
- Orders are strings rather than structured product records.
- There is no database, payment, price calculation, or persistence.
- Production placement should use transactions and idempotency keys.
- Model tool selection should be traced and evaluated.

# 14. Troubleshooting

### Authentication error

Verify that `GOOGLE_API_KEY` is enabled for the notebook and has Gemini API
access.

### Model not found

Set `GEMINI_MODEL` to a model available to your account.

### Mermaid PNG fails

The helper prints Mermaid source as a fallback.

### Recursion-limit error

Verify that the graph can reach `END` before increasing the limit.

### Gemini talks instead of using a tool

Inspect `AIMessage.tool_calls`, strengthen the system instruction, keep
temperature at zero, and test realistic prompts.

### Mixed tool-group error

The system instruction prohibits menu and order calls in one response. The
explicit error prevents unmatched tool-call messages.

# Deliverables checklist

- [x] Required packages installed
- [x] Kaggle, Colab, and local API-key support
- [x] `OrderState` with `TypedDict`
- [x] `add_messages` documented
- [x] One-turn chatbot graph
- [x] Second manual turn
- [x] Human node and conditional exit
- [x] Live-menu tool and `ToolNode`
- [x] Order tool schemas
- [x] Trusted custom order node
- [x] Drink and stock validation
- [x] Confirmation and placement flow
- [x] Graph visualizations
- [x] Non-interactive smoke tests
- [x] Optional interactive run
- [x] Thorough code comments and Markdown documentation

# Conclusion

BaristaBot demonstrates the central LangGraph pattern:

```text
shared state
    + nodes that return updates
    + edges that control transitions
    + tools that expose controlled capabilities
```

Gemini interprets natural language and requests actions, while trusted
Python code validates and changes application state.

# References

- LangGraph overview:
  https://docs.langchain.com/oss/python/langgraph/overview
- LangGraph Graph API:
  https://docs.langchain.com/oss/python/langgraph/graph-api
- LangChain Gemini integration:
  https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- Gemini models:
  https://ai.google.dev/gemini-api/docs/models
- Google AI Studio:
  https://aistudio.google.com/